In [1]:
rebuild_dimensions = True    # dimensions are small; rebuilding is cheap and safe

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Source tables:")
spark.sql("SHOW TABLES").show()

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 4, Finished, Available, Finished, False)

Source tables:
+--------------------+-----------------+-----------+
|           namespace|        tableName|isTemporary|
+--------------------+-----------------+-----------+
|`Cargo-Ops-Analyt...|    gold_dim_date|      false|
|`Cargo-Ops-Analyt...|   gold_dim_route|      false|
|`Cargo-Ops-Analyt...|gold_fact_flights|      false|
|`Cargo-Ops-Analyt...|gold_fact_tonnage|      false|
|`Cargo-Ops-Analyt...|   pipeline_state|      false|
|`Cargo-Ops-Analyt...|   silver_flights|      false|
|`Cargo-Ops-Analyt...|   silver_tonnage|      false|
+--------------------+-----------------+-----------+



In [3]:
# Build the date dimension from the actual data range, not a hardcoded span,
# so it always covers exactly what we have — no orphan facts, no empty years.
bounds = spark.sql("""
    SELECT MIN(d) AS min_d, MAX(d) AS max_d FROM (
        SELECT MIN(flight_date) AS d FROM silver_flights
        UNION ALL SELECT MAX(flight_date) FROM silver_flights
        UNION ALL SELECT MIN(flight_date) FROM silver_tonnage
        UNION ALL SELECT MAX(flight_date) FROM silver_tonnage
    )
""").collect()[0]

print(f"Date range: {bounds.min_d} to {bounds.max_d}")

df_date = (spark.sql(f"""
    SELECT explode(sequence(
        to_date('{bounds.min_d}'), to_date('{bounds.max_d}'), interval 1 day
    )) AS date_key
""")
    .withColumn("year",          F.year("date_key"))
    .withColumn("quarter_no",    F.quarter("date_key"))
    .withColumn("quarter",       F.concat(F.lit("Q"), F.quarter("date_key")))
    .withColumn("month_no",      F.month("date_key"))
    .withColumn("month_name",    F.date_format("date_key", "MMM"))
    .withColumn("month_year",    F.date_format("date_key", "MMM yyyy"))
    .withColumn("day_of_month",  F.dayofmonth("date_key"))
    .withColumn("day_name",      F.date_format("date_key", "EEE"))
    # Monday=1 .. Sunday=7. Spark's dayofweek is Sunday=1, so shift it.
    .withColumn("day_of_week",   ((F.dayofweek("date_key") + 5) % 7) + 1)
    .withColumn("week_of_year",  F.weekofyear("date_key"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("date_key").isin(1, 7), True).otherwise(False))
)

df_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_dim_date")

print(f"gold_dim_date: {df_date.count():,} rows")
df_date.show(3)

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 5, Finished, Available, Finished, False)

Date range: 2015-01-01 to 2017-12-01
gold_dim_date: 1,066 rows
+----------+----+----------+-------+--------+----------+----------+------------+--------+-----------+------------+----------+
|  date_key|year|quarter_no|quarter|month_no|month_name|month_year|day_of_month|day_name|day_of_week|week_of_year|is_weekend|
+----------+----+----------+-------+--------+----------+----------+------------+--------+-----------+------------+----------+
|2015-01-01|2015|         1|     Q1|       1|       Jan|  Jan 2015|           1|     Thu|          4|           1|     false|
|2015-01-02|2015|         1|     Q1|       1|       Jan|  Jan 2015|           2|     Fri|          5|           1|     false|
|2015-01-03|2015|         1|     Q1|       1|       Jan|  Jan 2015|           3|     Sat|          6|           1|      true|
+----------+----+----------+-------+--------+----------+----------+------------+--------+-----------+------------+----------+
only showing top 3 rows



In [8]:
# Conformed dimension: the SAME carrier list serves both facts. That's what
# makes a carrier slicer filter flights and tonnage coherently.
#
# Union both sources because they barely overlap — freighters appear only in
# tonnage, and some passenger carriers only in flights. Taking either alone
# would silently drop rows from the other fact.
carriers_flights = spark.sql("""
    SELECT DISTINCT carrier, CAST(NULL AS STRING) AS carrier_name,
           TRUE AS in_flights, FALSE AS in_tonnage
    FROM silver_flights WHERE carrier IS NOT NULL
""")

carriers_tonnage = spark.sql("""
    SELECT DISTINCT carrier, FIRST(carrier_name) AS carrier_name,
           FALSE AS in_flights, TRUE AS in_tonnage
    FROM silver_tonnage WHERE carrier IS NOT NULL
    GROUP BY carrier
""")

df_carrier = (carriers_flights.unionByName(carriers_tonnage)
    .groupBy("carrier")
    .agg(
        F.max("carrier_name").alias("carrier_name"),
        F.max("in_flights").alias("in_flights"),
        F.max("in_tonnage").alias("in_tonnage"),
    )
    # Flag the overlap explicitly — this IS the "two businesses" insight,
    # made queryable rather than left implicit.
    .withColumn("carrier_type",
        F.when(F.col("in_flights") & F.col("in_tonnage"), "Both")
         .when(F.col("in_tonnage"), "Cargo only")
         .otherwise("Flights only"))
    .withColumn("carrier_name", F.coalesce(F.col("carrier_name"), F.col("carrier")))
)

df_carrier.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_dim_carrier")

print(f"gold_dim_carrier: {df_carrier.count():,} rows")
df_carrier.groupBy("carrier_type").count().show()

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 11, Finished, Available, Finished, False)

gold_dim_carrier: 137 rows
+------------+-----+
|carrier_type|count|
+------------+-----+
|        Both|   14|
|  Cargo only|  123|
+------------+-----+



In [9]:
routes_flights = spark.sql("""
    SELECT DISTINCT route, origin, dest FROM silver_flights WHERE route IS NOT NULL
""")
routes_tonnage = spark.sql("""
    SELECT DISTINCT route, origin, dest FROM silver_tonnage WHERE route IS NOT NULL
""")

df_route = (routes_flights.unionByName(routes_tonnage)
    .dropDuplicates(["route"])
    .withColumn("is_domestic", F.lit(True))   # US_CARRIER_ONLY source
)

df_route.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_dim_route")

print(f"gold_dim_route: {df_route.count():,} rows")

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 12, Finished, Available, Finished, False)

gold_dim_route: 49,324 rows


In [10]:
# The fact keeps its natural keys (carrier, route, date) rather than integer
# surrogates. In a Delta/DirectLake model this is fine — Power BI relates on
# these directly, and it keeps the table readable in SQL.
df_fact_flights = spark.sql("""
    SELECT
        flight_date, year, month,
        carrier, route, origin, dest,
        flight_number, tail_number,
        dep_delay, arr_delay, arr_delay_missing,
        cancelled, diverted, distance_miles,
        on_time, delayed,
        delay_cause,
        carrier_delay_min, weather_delay_min, nas_delay_min,
        security_delay_min, late_aircraft_delay_min,
        delay_severity_band, delay_severity_sort, recovered_in_flight
    FROM silver_flights
""")

df_fact_flights.write.format("delta").mode("overwrite") \
    .partitionBy("year", "month") \
    .option("overwriteSchema", "true").saveAsTable("gold_fact_flights")

print(f"gold_fact_flights: {df_fact_flights.count():,} rows")

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 13, Finished, Available, Finished, False)

gold_fact_flights: 1,888,622 rows


In [16]:
# Load Factor Band and Yield Tier were DAX calculated columns. DirectLake
# doesn't support those, so they're computed here and stored as real columns.
df_fact_tonnage = (spark.table("silver_tonnage")
    # Drop zero-activity rows AND rows with no carrier. T-100 includes many
    # carrier/aircraft combinations with no operations, plus a few hundred
    # null-carrier records with no freight — the latter would orphan against
    # dim_carrier and break the relationship.
    .filter((F.col("departures") > 0) & F.col("carrier").isNotNull())

    .withColumn("row_load_factor", F.when(F.col("aftk") > 0,
                                          F.col("ftk") / F.col("aftk")))

    .withColumn("load_factor_band",
        F.when(F.col("aftk") <= 0,             "No Capacity")
         .when(F.col("row_load_factor") < 0.5,  "Underutilized (<50%)")
         .when(F.col("row_load_factor") < 0.7,  "Moderate (50-70%)")
         .when(F.col("row_load_factor") < 0.85, "Healthy (70-85%)")
         .otherwise("High (85%+)"))

    # Integer companion so visuals sort logically, not alphabetically.
    .withColumn("load_factor_band_sort",
        F.when(F.col("aftk") <= 0,             0)
         .when(F.col("row_load_factor") < 0.5,  1)
         .when(F.col("row_load_factor") < 0.7,  2)
         .when(F.col("row_load_factor") < 0.85, 3)
         .otherwise(4))

    .withColumn("is_mail_dominant",
        F.when(F.col("mail_tonnes") > F.col("freight_tonnes"), 1).otherwise(0))
)

df_fact_tonnage.write.format("delta").mode("overwrite") \
    .partitionBy("year", "month") \
    .option("overwriteSchema", "true").saveAsTable("gold_fact_tonnage")

print(f"gold_fact_tonnage: {df_fact_tonnage.count():,} rows")
df_fact_tonnage.groupBy("load_factor_band", "load_factor_band_sort") \
    .count().orderBy("load_factor_band_sort").show()

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 19, Finished, Available, Finished, False)

gold_fact_tonnage: 1,257,815 rows
+--------------------+---------------------+-------+
|    load_factor_band|load_factor_band_sort|  count|
+--------------------+---------------------+-------+
|         No Capacity|                    0|   3385|
|Underutilized (<50%)|                    1|1199113|
|   Moderate (50-70%)|                    2|  40219|
|    Healthy (70-85%)|                    3|  10162|
|         High (85%+)|                    4|   4936|
+--------------------+---------------------+-------+



In [17]:
print("=" * 60)
print("GOLD LAYER VALIDATION")
print("=" * 60)

# Orphan check: every fact row must find its dimension. A non-zero count here
# means a broken relationship in Power BI and wrong totals in every visual.
checks = {
 "flights w/o carrier": """
    SELECT COUNT(*) c FROM gold_fact_flights f
    LEFT JOIN gold_dim_carrier d ON f.carrier = d.carrier WHERE d.carrier IS NULL""",
 "flights w/o route": """
    SELECT COUNT(*) c FROM gold_fact_flights f
    LEFT JOIN gold_dim_route d ON f.route = d.route WHERE d.route IS NULL""",
 "flights w/o date": """
    SELECT COUNT(*) c FROM gold_fact_flights f
    LEFT JOIN gold_dim_date d ON f.flight_date = d.date_key WHERE d.date_key IS NULL""",
 "tonnage w/o carrier": """
    SELECT COUNT(*) c FROM gold_fact_tonnage f
    LEFT JOIN gold_dim_carrier d ON f.carrier = d.carrier WHERE d.carrier IS NULL""",
 "tonnage w/o route": """
    SELECT COUNT(*) c FROM gold_fact_tonnage f
    LEFT JOIN gold_dim_route d ON f.route = d.route WHERE d.route IS NULL""",
 "tonnage w/o date": """
    SELECT COUNT(*) c FROM gold_fact_tonnage f
    LEFT JOIN gold_dim_date d ON f.flight_date = d.date_key WHERE d.date_key IS NULL""",
}

failed = 0
for label, sql in checks.items():
    n = spark.sql(sql).collect()[0].c
    status = "OK" if n == 0 else "FAIL"
    if n: failed += 1
    print(f"  {status:5s} {label}: {n:,} orphans")

print("=" * 60)
print("ALL CHECKS PASSED" if not failed else f"{failed} CHECK(S) FAILED")

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 20, Finished, Available, Finished, False)

GOLD LAYER VALIDATION
  OK    flights w/o carrier: 0 orphans
  OK    flights w/o route: 0 orphans
  OK    flights w/o date: 0 orphans
  OK    tonnage w/o carrier: 0 orphans
  OK    tonnage w/o route: 0 orphans
  OK    tonnage w/o date: 0 orphans
ALL CHECKS PASSED


In [13]:
spark.sql("""
    SELECT t.carrier, COUNT(*) AS rows,
           ROUND(SUM(t.freight_tonnes), 2) AS freight_tonnes,
           MIN(t.year) AS first_year, MAX(t.year) AS last_year
    FROM gold_fact_tonnage t
    LEFT JOIN gold_dim_carrier d ON t.carrier = d.carrier
    WHERE d.carrier IS NULL
    GROUP BY t.carrier ORDER BY rows DESC
""").show(50, truncate=False)

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 16, Finished, Available, Finished, False)

+-------+----+--------------+----------+---------+
|carrier|rows|freight_tonnes|first_year|last_year|
+-------+----+--------------+----------+---------+
|NULL   |325 |0.0           |2015      |2015     |
+-------+----+--------------+----------+---------+



In [18]:
spark.sql("""
    SELECT d.year,
           COUNT(*)                                                    AS flights,
           ROUND(100.0*SUM(f.on_time)/SUM(f.on_time+f.delayed), 1)     AS otp_pct
    FROM gold_fact_flights f JOIN gold_dim_date d ON f.flight_date = d.date_key
    GROUP BY d.year ORDER BY d.year
""").show()

spark.sql("""
    SELECT c.carrier_type, COUNT(DISTINCT c.carrier) AS carriers,
           ROUND(SUM(t.freight_tonnes), 0)           AS freight_tonnes
    FROM gold_fact_tonnage t JOIN gold_dim_carrier c ON t.carrier = c.carrier
    GROUP BY c.carrier_type ORDER BY freight_tonnes DESC
""").show()

StatementMeta(, 598fcc27-f07a-400e-be80-4bca681bc19d, 21, Finished, Available, Finished, False)

+----+-------+-------+
|year|flights|otp_pct|
+----+-------+-------+
|2015|1888622|   80.0|
+----+-------+-------+

+------------+--------+--------------+
|carrier_type|carriers|freight_tonnes|
+------------+--------+--------------+
|  Cargo only|     122|   3.7885621E7|
|        Both|      14|     5195029.0|
+------------+--------+--------------+

